# Interactive graph visualization

Staged views of a measurement-based CV computation with `photographiq.interactive`.
Python is the authoritative model: every number below is computed by PhotoGraphiQ; the
browser (Cytoscape.js / Plotly.js from pinned CDNs) only displays and edits.
Pages are shown inline as `srcdoc` iframes; `.show()` opens them in a browser and
`.save_html(path)` writes a standalone file. Static figures use Matplotlib only.

In [1]:
import json
import tempfile
from pathlib import Path

import photographiq as pg
from photographiq.cvflow import flow_pattern
from photographiq.flows import analyze
from photographiq.interactive import (
    GraphView,
    apply_edits,
    covariance_heatmap,
    execution_trace,
    squeezing_ellipses,
    wigner_plot,
)
from photographiq.ir import OpenGraph

graph = pg.CVGraph.line(4, squeezing=1.5, inputs=(0,), outputs=(3,))
shears = {0: 0.3, 1: -0.5, 2: 0.0}
open_graph = OpenGraph.from_cvgraph(graph, shears)
flow_result = analyze(open_graph)  # CV flow search + certificate + determinism check
pattern = flow_pattern(graph, flow_result.measurement_order, shears=shears)
result = pg.simulate(pattern, backend="gaussian", seed=7)
result.outcomes

{0: 0.005422046312954974, 1: 1.2441923697526902, 2: -1.3121492010207545}

## Stage A: construct (graph, weights, roles, squeezing; no flow arrows)

In [2]:
view = GraphView(open_graph, title="Four-node CV wire").construct()
view

GraphView(graph=OpenGraph(model=<Model.CV_GAUSSIAN: 'cv-gaussian'>, graph=<networkx.classes.graph.Graph object at 0x000001E5CC4F0B90>, inputs=(0,), outputs=(3,), measurements={0: HomodyneSpec(shear=0.3), 1: HomodyneSpec(shear=-0.5), 2: HomodyneSpec(shear=0.0)}, squeezing={0: 1.5, 1: 1.5, 2: 1.5, 3: 1.5}), layout=Layout(positions={0: (0.0, 0.0), 1: (1.0, 0.0), 2: (2.0, 0.0), 3: (3.0, 0.0)}), stage='construct', flow=None, corrections=None, trace=None, annotations={}, notes=(), title='Four-node CV wire')

## Stage B: flow overlay

`with_flow` takes the `photographiq.flows.FlowAnalysisResult` (or any object/dict with the
same attributes): certification status, algorithm, assumptions, measurement order,
layers and dependency arrows derived from its `correction_map`.

In [3]:
print(flow_result.status, flow_result.algorithm, flow_result.measurement_order)
flow_view = view.with_flow(flow_result)
sorted(flow_view.flow_arrows())

certified cv-flow (Booth-Markham 2023 real-linear condition; layered order search) (0, 1, 2)


[(0, 1), (0, 2), (1, 2), (1, 3), (2, 3)]

In [4]:
flow_view

GraphView(graph=OpenGraph(model=<Model.CV_GAUSSIAN: 'cv-gaussian'>, graph=<networkx.classes.graph.Graph object at 0x000001E5D165F0B0>, inputs=(0,), outputs=(3,), measurements={0: HomodyneSpec(shear=0.3), 1: HomodyneSpec(shear=-0.5), 2: HomodyneSpec(shear=0.0)}, squeezing={0: 1.5, 1: 1.5, 2: 1.5, 3: 1.5}), layout=Layout(positions={0: (0.0, 0.0), 1: (1.0, 0.0), 2: (2.0, 0.0), 3: (3.0, 0.0)}), stage='flow', flow=FlowOverlay(status='certified', algorithm='cv-flow (Booth-Markham 2023 real-linear condition; layered order search)', model='cv-gaussian', order=(0, 1, 2), layers=((0,), (1,), (2,)), corrections={1: Correction(target=1, kind='cv', x=(), z=(), q=((0, -1.044030650891055),), p=(), q0=0.0, p0=0.0), 2: Correction(target=2, kind='cv', x=(), z=(), q=((1, -1.118033988749895),), p=((0, -1.044030650891055),), q0=0.0, p0=0.0), 3: Correction(target=3, kind='cv', x=(), z=(), q=((2, -1.0),), p=((1, -1.118033988749895),), q0=0.0, p0=0.0)}, assumptions=('Booth-Markham real-linear CV flow for homodyne measurements on a weighted graph', "determinism is an infinite-squeezing statement; finite squeezing adds noise N and an outcome-dependent conditional shift G m, both in certificate['determinism']", 'unit-efficiency homodyne detection', 'missing resource squeezing defaults to r=1.0 (OpenGraph.to_cvgraph convention)'), certificate={'total_order': CVFlow(order=(0, 1, 2), corrections={0: {1: 1.0, 2: 0.0, 3: 0.0}, 1: {2: 1.0, 3: 0.0}, 2: {3: 1.0}}, residuals=(0.0, 0.0, 0.0)), 'layer_corrections': {2: {3: 1.0}, 1: {2: 1.0, 3: 0.0}, 0: {1: 1.0, 2: 0.0, 3: 0.0}}, 'shears': {0: 0.3, 1: -0.5, 2: 0.0}, 'checks': ('layered real-linear condition', 'layered vectors re-checked on the flattened total order', 'certify_cv_flow on the total order', 'ideal-limit determinism by exact Gaussian postselection'), 'regimes': {'ideal_limit': 'deterministic unitary: float64 decay check (shift=3.0)', 'finite_squeezing': 'outcome-averaged Gaussian channel (S, N) at the given squeezing (min r = 1.5), S the ideal map; residual outcome dependence 0.189 (float64)'}, 'determinism': {'matrix': array([[ 0.3 ,  1.  ],
       [-1.15, -0.5 ]]), 'noise': array([[ 4.97870684e-02, -3.04857870e-18],
       [-3.04857870e-18,  9.95741367e-02]]), 'displacement': array([0., 0.]), 'outputs': (3,), 'symplectic_residual': 2.220446049250313e-16, 'finite_squeezing_noise': array([[ 4.97870684e-02, -3.04857870e-18],
       [-3.04857870e-18,  9.95741367e-02]]), 'ideal_singular_noise': array([[0., 0.],
       [0., 0.]]), 'ideal_singular_directions': 0, 'expected_singular_directions': 0, 'noise_model_residual': 2.938735877055719e-39, 'finite_squeezing_gain': 0.1892891853018888, 'finite_squeezing_conditional_map_error': 0.5347200305080877, 'finite_squeezing_outcome_spread': 0.550274421193717, 'levels': ({'extra_squeezing': 0.0, 'matrix_change': 0.0, 'gain': 0.1892891853018888, 'conditional_map_error': 0.5347200305080877, 'covariance_outcome_dependence': 0.0, 'spread_min': 0.016713291247678966, 'finite_spread_max': 0.550274421193717, 'spread_rounding': np.float64(2.5006310480364605e-13)}, {'extra_squeezing': 3.0, 'matrix_change': 0.0, 'gain': 0.0005616249995081019, 'conditional_map_error': 0.0015764511250253577, 'covariance_outcome_dependence': 0.0, 'spread_min': 3.6892918028831866e-05, 'finite_spread_max': 0.0016074122247982437, 'spread_rounding': np.float64(1.0075778065214892e-10)}, {'extra_squeezing': 6.0, 'matrix_change': 0.0, 'gain': 1.3928081505648087e-06, 'conditional_map_error': 3.909476864905059e-06, 'covariance_outcome_dependence': 0.0, 'spread_min': 9.125525824306282e-08, 'finite_spread_max': 3.987278898185e-06, 'spread_rounding': np.float64(4.0648589573965394e-08)}), 'decay_exponents': {'gain': 0.9999187341886869, 'conditional_map_error': 0.9999213175660672}, 'tolerance': 1.1500000000000002e-09, 'shift': 3.0, 'failed_checks': (), 'ideal_limit_deterministic': True}}, diagnostics={'complete': True, 'rank_decisions': {'rule': 'singular values of the 2^k-equilibrated, normalised correction m

## Stage C: corrections (displacements as linear combinations of outcomes m_k)

In [5]:
correction_view = flow_view.with_corrections()
for target, correction in correction_view.corrections.items():
    print(target, correction.formula())
correction_view

1 q += -1.044 m[0]
2 q += -1.118 m[1]; p += -1.044 m[0]
3 q += -1 m[2]; p += -1.118 m[1]


GraphView(graph=OpenGraph(model=<Model.CV_GAUSSIAN: 'cv-gaussian'>, graph=<networkx.classes.graph.Graph object at 0x000001E5CF60A690>, inputs=(0,), outputs=(3,), measurements={0: HomodyneSpec(shear=0.3), 1: HomodyneSpec(shear=-0.5), 2: HomodyneSpec(shear=0.0)}, squeezing={0: 1.5, 1: 1.5, 2: 1.5, 3: 1.5}), layout=Layout(positions={0: (0.0, 0.0), 1: (1.0, 0.0), 2: (2.0, 0.0), 3: (3.0, 0.0)}), stage='corrections', flow=FlowOverlay(status='certified', algorithm='cv-flow (Booth-Markham 2023 real-linear condition; layered order search)', model='cv-gaussian', order=(0, 1, 2), layers=((0,), (1,), (2,)), corrections={1: Correction(target=1, kind='cv', x=(), z=(), q=((0, -1.044030650891055),), p=(), q0=0.0, p0=0.0), 2: Correction(target=2, kind='cv', x=(), z=(), q=((1, -1.118033988749895),), p=((0, -1.044030650891055),), q0=0.0, p0=0.0), 3: Correction(target=3, kind='cv', x=(), z=(), q=((2, -1.0),), p=((1, -1.118033988749895),), q0=0.0, p0=0.0)}, assumptions=('Booth-Markham real-linear CV flow for homodyne measurements on a weighted graph', "determinism is an infinite-squeezing statement; finite squeezing adds noise N and an outcome-dependent conditional shift G m, both in certificate['determinism']", 'unit-efficiency homodyne detection', 'missing resource squeezing defaults to r=1.0 (OpenGraph.to_cvgraph convention)'), certificate={'total_order': CVFlow(order=(0, 1, 2), corrections={0: {1: 1.0, 2: 0.0, 3: 0.0}, 1: {2: 1.0, 3: 0.0}, 2: {3: 1.0}}, residuals=(0.0, 0.0, 0.0)), 'layer_corrections': {2: {3: 1.0}, 1: {2: 1.0, 3: 0.0}, 0: {1: 1.0, 2: 0.0, 3: 0.0}}, 'shears': {0: 0.3, 1: -0.5, 2: 0.0}, 'checks': ('layered real-linear condition', 'layered vectors re-checked on the flattened total order', 'certify_cv_flow on the total order', 'ideal-limit determinism by exact Gaussian postselection'), 'regimes': {'ideal_limit': 'deterministic unitary: float64 decay check (shift=3.0)', 'finite_squeezing': 'outcome-averaged Gaussian channel (S, N) at the given squeezing (min r = 1.5), S the ideal map; residual outcome dependence 0.189 (float64)'}, 'determinism': {'matrix': array([[ 0.3 ,  1.  ],
       [-1.15, -0.5 ]]), 'noise': array([[ 4.97870684e-02, -3.04857870e-18],
       [-3.04857870e-18,  9.95741367e-02]]), 'displacement': array([0., 0.]), 'outputs': (3,), 'symplectic_residual': 2.220446049250313e-16, 'finite_squeezing_noise': array([[ 4.97870684e-02, -3.04857870e-18],
       [-3.04857870e-18,  9.95741367e-02]]), 'ideal_singular_noise': array([[0., 0.],
       [0., 0.]]), 'ideal_singular_directions': 0, 'expected_singular_directions': 0, 'noise_model_residual': 2.938735877055719e-39, 'finite_squeezing_gain': 0.1892891853018888, 'finite_squeezing_conditional_map_error': 0.5347200305080877, 'finite_squeezing_outcome_spread': 0.550274421193717, 'levels': ({'extra_squeezing': 0.0, 'matrix_change': 0.0, 'gain': 0.1892891853018888, 'conditional_map_error': 0.5347200305080877, 'covariance_outcome_dependence': 0.0, 'spread_min': 0.016713291247678966, 'finite_spread_max': 0.550274421193717, 'spread_rounding': np.float64(2.5006310480364605e-13)}, {'extra_squeezing': 3.0, 'matrix_change': 0.0, 'gain': 0.0005616249995081019, 'conditional_map_error': 0.0015764511250253577, 'covariance_outcome_dependence': 0.0, 'spread_min': 3.6892918028831866e-05, 'finite_spread_max': 0.0016074122247982437, 'spread_rounding': np.float64(1.0075778065214892e-10)}, {'extra_squeezing': 6.0, 'matrix_change': 0.0, 'gain': 1.3928081505648087e-06, 'conditional_map_error': 3.909476864905059e-06, 'covariance_outcome_dependence': 0.0, 'spread_min': 9.125525824306282e-08, 'finite_spread_max': 3.987278898185e-06, 'spread_rounding': np.float64(4.0648589573965394e-08)}), 'decay_exponents': {'gain': 0.9999187341886869, 'conditional_map_error': 0.9999213175660672}, 'tolerance': 1.1500000000000002e-09, 'shift': 3.0, 'failed_checks': (), 'ideal_limit_deterministic': True}}, diagnostics={'complete': True, 'rank_decisions': {'rule': 'singular values of the 2^k-equilibrated, normalised corre

## Stage D: execution with the recorded outcomes (Play / Step / Reset)

In [6]:
trace = execution_trace(pattern, result)
for step in trace.steps:
    print(step.index, step.kind, step.label)
execution_view = correction_view.with_execution(trace)
execution_view

0 prepare Prepare 1 (squeezing=1.5)
1 prepare Prepare 2 (squeezing=1.5)
2 prepare Prepare 3 (squeezing=1.5)
3 entangle Entangle 0, 1 (weight=1)
4 entangle Entangle 1, 2 (weight=1)
5 entangle Entangle 2, 3 (weight=1)
6 measure Measure[Homodyne] 0 (angle=1.279) -> 0 = 0.005422
7 feed-forward Displace 1 (q=-0.005661, p=0)
8 feed-forward Displace 2 (q=0, p=-0.005661)
9 measure Measure[Homodyne] 1 (angle=2.034) -> 1 = 1.244
10 feed-forward Displace 2 (q=-1.391, p=0)
11 feed-forward Displace 3 (q=0, p=-1.391)
12 measure Measure[Homodyne] 2 (angle=1.571) -> 2 = -1.312
13 feed-forward Displace 3 (q=1.312, p=0)
14 output Output (3,) (implicit)


GraphView(graph=OpenGraph(model=<Model.CV_GAUSSIAN: 'cv-gaussian'>, graph=<networkx.classes.graph.Graph object at 0x000001E5D167EEA0>, inputs=(0,), outputs=(3,), measurements={0: HomodyneSpec(shear=0.3), 1: HomodyneSpec(shear=-0.5), 2: HomodyneSpec(shear=0.0)}, squeezing={0: 1.5, 1: 1.5, 2: 1.5, 3: 1.5}), layout=Layout(positions={0: (0.0, 0.0), 1: (1.0, 0.0), 2: (2.0, 0.0), 3: (3.0, 0.0)}), stage='execution', flow=FlowOverlay(status='certified', algorithm='cv-flow (Booth-Markham 2023 real-linear condition; layered order search)', model='cv-gaussian', order=(0, 1, 2), layers=((0,), (1,), (2,)), corrections={1: Correction(target=1, kind='cv', x=(), z=(), q=((0, -1.044030650891055),), p=(), q0=0.0, p0=0.0), 2: Correction(target=2, kind='cv', x=(), z=(), q=((1, -1.118033988749895),), p=((0, -1.044030650891055),), q0=0.0, p0=0.0), 3: Correction(target=3, kind='cv', x=(), z=(), q=((2, -1.0),), p=((1, -1.118033988749895),), q0=0.0, p0=0.0)}, assumptions=('Booth-Markham real-linear CV flow for homodyne measurements on a weighted graph', "determinism is an infinite-squeezing statement; finite squeezing adds noise N and an outcome-dependent conditional shift G m, both in certificate['determinism']", 'unit-efficiency homodyne detection', 'missing resource squeezing defaults to r=1.0 (OpenGraph.to_cvgraph convention)'), certificate={'total_order': CVFlow(order=(0, 1, 2), corrections={0: {1: 1.0, 2: 0.0, 3: 0.0}, 1: {2: 1.0, 3: 0.0}, 2: {3: 1.0}}, residuals=(0.0, 0.0, 0.0)), 'layer_corrections': {2: {3: 1.0}, 1: {2: 1.0, 3: 0.0}, 0: {1: 1.0, 2: 0.0, 3: 0.0}}, 'shears': {0: 0.3, 1: -0.5, 2: 0.0}, 'checks': ('layered real-linear condition', 'layered vectors re-checked on the flattened total order', 'certify_cv_flow on the total order', 'ideal-limit determinism by exact Gaussian postselection'), 'regimes': {'ideal_limit': 'deterministic unitary: float64 decay check (shift=3.0)', 'finite_squeezing': 'outcome-averaged Gaussian channel (S, N) at the given squeezing (min r = 1.5), S the ideal map; residual outcome dependence 0.189 (float64)'}, 'determinism': {'matrix': array([[ 0.3 ,  1.  ],
       [-1.15, -0.5 ]]), 'noise': array([[ 4.97870684e-02, -3.04857870e-18],
       [-3.04857870e-18,  9.95741367e-02]]), 'displacement': array([0., 0.]), 'outputs': (3,), 'symplectic_residual': 2.220446049250313e-16, 'finite_squeezing_noise': array([[ 4.97870684e-02, -3.04857870e-18],
       [-3.04857870e-18,  9.95741367e-02]]), 'ideal_singular_noise': array([[0., 0.],
       [0., 0.]]), 'ideal_singular_directions': 0, 'expected_singular_directions': 0, 'noise_model_residual': 2.938735877055719e-39, 'finite_squeezing_gain': 0.1892891853018888, 'finite_squeezing_conditional_map_error': 0.5347200305080877, 'finite_squeezing_outcome_spread': 0.550274421193717, 'levels': ({'extra_squeezing': 0.0, 'matrix_change': 0.0, 'gain': 0.1892891853018888, 'conditional_map_error': 0.5347200305080877, 'covariance_outcome_dependence': 0.0, 'spread_min': 0.016713291247678966, 'finite_spread_max': 0.550274421193717, 'spread_rounding': np.float64(2.5006310480364605e-13)}, {'extra_squeezing': 3.0, 'matrix_change': 0.0, 'gain': 0.0005616249995081019, 'conditional_map_error': 0.0015764511250253577, 'covariance_outcome_dependence': 0.0, 'spread_min': 3.6892918028831866e-05, 'finite_spread_max': 0.0016074122247982437, 'spread_rounding': np.float64(1.0075778065214892e-10)}, {'extra_squeezing': 6.0, 'matrix_change': 0.0, 'gain': 1.3928081505648087e-06, 'conditional_map_error': 3.909476864905059e-06, 'covariance_outcome_dependence': 0.0, 'spread_min': 9.125525824306282e-08, 'finite_spread_max': 3.987278898185e-06, 'spread_rounding': np.float64(4.0648589573965394e-08)}), 'decay_exponents': {'gain': 0.9999187341886869, 'conditional_map_error': 0.9999213175660672}, 'tolerance': 1.1500000000000002e-09, 'shift': 3.0, 'failed_checks': (), 'ideal_limit_deterministic': True}}, diagnostics={'complete': True, 'rank_decisions': {'rule': 'singular values of the 2^k-equilibrated, normalised correct

## Static manuscript export and the edit round trip

In [7]:
out = Path(tempfile.mkdtemp())
for stage in ("construct", "flow", "corrections", "execution"):
    execution_view.save(out / f"wire_{stage}.svg", stage=stage)
execution_view.save_html(out / "wire.html")
# The page's Export button produces exactly this structure; Python re-validates it.
exported = json.dumps({"graph": view.graph.to_dict(), "layout": view.layout.to_dict()})
edited, layout = apply_edits(exported)
assert edited.to_dict() == view.graph.to_dict()
sorted(p.name for p in out.iterdir())

['wire.html',
 'wire_construct.svg',
 'wire_corrections.svg',
 'wire_execution.svg',
 'wire_flow.svg']

## Scientific plots of the output state

In [8]:
state = result.state
covariance_heatmap(state)

Plot(traces=[{'type': 'heatmap', 'x': ['q[3]', 'p[3]'], 'y': ['q[3]', 'p[3]'], 'z': [[1.0014849064017888, -0.7057244259586516], [-0.7057244259586516, 1.4958258040822219]], 'zmid': 0, 'zmin': -1.4958258040822219, 'zmax': 1.4958258040822219, 'colorscale': [[0.0, '#2166ac'], [0.5, '#f7f7f7'], [1.0, '#b2182b']], 'colorbar': {'title': 'V'}}], layout={'title': {'text': 'Covariance matrix (vacuum V = I)'}, 'xaxis': {'title': {'text': ''}}, 'yaxis': {'title': {'text': ''}, 'autorange': 'reversed'}, 'margin': {'t': 50}})

In [9]:
squeezing_ellipses(state)

Plot(traces=[{'type': 'scatter', 'mode': 'lines', 'x': [-0.4626953756836748, -0.4908728546836223, -0.5183126950344501, -0.5449814655174736, -0.5708466743438689, -0.5958768087409105, -0.6200413733454329, -0.6433109273577307, -0.6656571204106402, -0.6870527271100936, -0.7074716802050681, -0.726889102346516, -0.7452813363965829, -0.762625974251185, -0.7789018841408332, -0.7940892363764382, -0.8081695275087339, -0.8211256028718781, -0.8329416774837708, -0.8436033552776243, -0.8530976466413523, -0.8614129842434123, -0.8685392371258147, -0.8744677230471379, -0.8791912190605002, -0.8827039703136077, -0.8850016970601566, -0.8860815998740439, -0.8859423630600369, -0.8845841562567459, -0.882008634229945, -0.878218934856495, -0.8732196753013242, -0.8670169463921233, -0.8596183051986114, -0.8510327658254094, -0.8412707884297425, -0.83034426647735, -0.8182665122521291, -0.8050522406371682, -0.7907175511869271, -0.7752799085124117, -0.758758121003234, -0.7411723179124879, -0.7225439248323544, -0.7028956375903171, -0.6822513945977942, -0.6606363476848699, -0.6380768314566652, -0.6146003312086771, -0.5902354494401798, -0.5650118710064862, -0.538960326952523, -0.5121125570717878, -0.4845012712362978, -0.45616010954465125, -0.427123601336748, -0.3974271231251083, -0.3671068554940413, -0.3361997390191761, -0.30474342926106146, -0.272776250887664, -0.2403371509816653, -0.2074656515894409, -0.17420180156953374, -0.14058612779928992, -0.10665958579910081, -0.07246350983441027, -0.038039562556279854, -0.003429684241865863, 0.03132395830334965, 0.0661790231193132, 0.10109304467855758, 0.1360234856239057, 0.1709277885936876, 0.2057634280713302, 0.24048796219614887, 0.275059084472217, 0.309434675312314, 0.3435728533541559, 0.3774320264863814, 0.4109709425221349, 0.4441487394585005, 0.47692499526055915, 0.5092597771094114, 0.5411136900541653, 0.5724479250086175, 0.603224306034144, 0.6334053368512041, 0.6629542465227782, 0.6918350342540924, 0.72001251325404, 0.7474523536048677, 0.7741211240878915, 0.7999863329142866, 0.8250164673113284, 0.8491810319158507, 0.8724505859281483, 0.8947967789810578, 0.9161923856805112, 0.9366113387754857, 0.9560287609169338, 0.9744209949670006, 0.9917656328216028, 1.0080415427112508, 1.0232288949468562, 1.0373091860791517, 1.0502652614422956, 1.0620813360541885, 1.0727430138480418, 1.0822373052117702, 1.09055264281383, 1.0976788956962324, 1.1036073816175556, 1.1083308776309178, 1.1118436288840254, 1.1141413556305744, 1.1152212584444614, 1.1150820216304544, 1.1137238148271638, 1.1111482928003629, 1.1073585934269128, 1.1023593338717417, 1.096156604962541, 1.0887579637690292, 1.080172424395827, 1.0704104470001603, 1.0594839250477674, 1.0474061708225468, 1.034191899207586, 1.0198572097573448, 1.0044195670828295, 0.9878977795736518, 0.9703119764829059, 0.9516835834027719, 0.9320352961607349, 0.9113910531682117, 0.8897760062552879, 0.8672164900270833, 0.8437399897790948, 0.8193751080105978, 0.7941515295769037, 0.7680999855229411, 0.7412522156422052, 0.7136409298067157, 0.6852997681150687, 0.6562632599071658, 0.6265667816955266, 0.5962465140644592, 0.5653393975895944, 0.5338830878314791, 0.5019159094580818, 0.46947680955208276, 0.4366053101598588, 0.40334146013995126, 0.36972578636970777, 0.33579924436951913, 0.30160316840482804, 0.26717922112669823, 0.2325693428122833, 0.1978157002670682, 0.16296063545110423, 0.12804661389186028, 0.09311617294651173, 0.05821186997673026, 0.023376230499088227, -0.011348303625731082, -0.04591942590179876, -0.08029501674189668, -0.11443319478373806, -0.148292367915964, -0.181831283951717, -0.21500908088808296, -0.2477853366901413, -0.2801201185389931, -0.3119740314837475, -0.3433082664381992, -0.3740846474637265, -0.4042656782807863, -0.43381458795236083, -0.4626953756836746], 'y': [-0.5559175382693847, -0.5154478537782411, -0.474528602543447, -0.43320963836950455, -0.3915413020495614, -0.3495743600329081, -0.30735994257388, -0.26494948143751845, -0.22239464723788743, -0.1797472864853895, -0.13705935

In [10]:
wigner_plot(state, 3, points=81, extent=8)

Plot(traces=[{'type': 'heatmap', 'x': [-8.0, -7.8, -7.6, -7.4, -7.2, -7.0, -6.8, -6.6, -6.4, -6.2, -6.0, -5.8, -5.6, -5.4, -5.199999999999999, -5.0, -4.8, -4.6, -4.4, -4.199999999999999, -4.0, -3.8, -3.5999999999999996, -3.3999999999999995, -3.1999999999999993, -3.0, -2.8, -2.5999999999999996, -2.3999999999999995, -2.1999999999999993, -2.0, -1.7999999999999998, -1.5999999999999996, -1.3999999999999995, -1.1999999999999993, -1.0, -0.7999999999999998, -0.5999999999999996, -0.39999999999999947, -0.1999999999999993, 0.0, 0.20000000000000107, 0.40000000000000036, 0.5999999999999996, 0.8000000000000007, 1.0, 1.200000000000001, 1.4000000000000004, 1.6000000000000014, 1.8000000000000007, 2.0, 2.200000000000001, 2.4000000000000004, 2.6000000000000014, 2.8000000000000007, 3.0, 3.200000000000001, 3.4000000000000004, 3.6000000000000014, 3.8000000000000007, 4.0, 4.200000000000001, 4.4, 4.600000000000001, 4.800000000000001, 5.0, 5.200000000000001, 5.4, 5.600000000000001, 5.800000000000001, 6.0, 6.200000000000001, 6.4, 6.600000000000001, 6.800000000000001, 7.0, 7.200000000000001, 7.4, 7.600000000000001, 7.800000000000001, 8.0], 'y': [-8.0, -7.8, -7.6, -7.4, -7.2, -7.0, -6.8, -6.6, -6.4, -6.2, -6.0, -5.8, -5.6, -5.4, -5.199999999999999, -5.0, -4.8, -4.6, -4.4, -4.199999999999999, -4.0, -3.8, -3.5999999999999996, -3.3999999999999995, -3.1999999999999993, -3.0, -2.8, -2.5999999999999996, -2.3999999999999995, -2.1999999999999993, -2.0, -1.7999999999999998, -1.5999999999999996, -1.3999999999999995, -1.1999999999999993, -1.0, -0.7999999999999998, -0.5999999999999996, -0.39999999999999947, -0.1999999999999993, 0.0, 0.20000000000000107, 0.40000000000000036, 0.5999999999999996, 0.8000000000000007, 1.0, 1.200000000000001, 1.4000000000000004, 1.6000000000000014, 1.8000000000000007, 2.0, 2.200000000000001, 2.4000000000000004, 2.6000000000000014, 2.8000000000000007, 3.0, 3.200000000000001, 3.4000000000000004, 3.6000000000000014, 3.8000000000000007, 4.0, 4.200000000000001, 4.4, 4.600000000000001, 4.800000000000001, 5.0, 5.200000000000001, 5.4, 5.600000000000001, 5.800000000000001, 6.0, 6.200000000000001, 6.4, 6.600000000000001, 6.800000000000001, 7.0, 7.200000000000001, 7.4, 7.600000000000001, 7.800000000000001, 8.0], 'z': [[7.39043243703054e-56, 2.4625182711778455e-54, 7.7286549121317975e-53, 2.2847738282952795e-51, 6.362054112954425e-50, 1.6686542017578716e-48, 4.122400463857492e-47, 9.59287652713123e-46, 2.102627669033872e-44, 4.341009536858752e-43, 8.441778429524924e-42, 1.546293757167255e-40, 2.6678714546077706e-39, 4.335634288571216e-38, 6.636746486635867e-37, 9.569134506705421e-36, 1.299585730451301e-34, 1.662463248640758e-33, 2.0031525158295503e-32, 2.27347835933688e-31, 2.4304264153289095e-30, 2.4473099342021335e-29, 2.321187984483891e-28, 2.0737027269584675e-27, 1.74500842482576e-26, 1.3831312751392957e-25, 1.0326285293075504e-24, 7.261723050684262e-24, 4.8100555076867423e-23, 3.0010645645615366e-22, 1.7636623097519692e-21, 9.762709650421072e-21, 5.09026224061939e-20, 2.4999123663447274e-19, 1.1564430677818742e-18, 5.0389328821127776e-18, 2.0680819039384023e-17, 7.99487613534365e-17, 2.9111902564646354e-16, 9.984912944708991e-16, 3.2257657079630794e-15, 9.816037369414958e-15, 2.8135486538335663e-14, 7.596044504767718e-14, 1.931681034068998e-13, 4.626985891579386e-13, 1.0439405782533004e-12, 2.2185449231760607e-12, 4.440946277660949e-12, 8.373319371623412e-12, 1.487081042350575e-11, 2.4876338768256277e-11, 3.919702735456957e-11, 5.8174766106794086e-11, 8.132629809457205e-11, 1.0708834492051174e-10, 1.3282146105435367e-10, 1.5517049454347284e-10, 1.7075163843449156e-10, 1.7698459209644976e-10, 1.727909065545131e-10, 1.5889899536916237e-10, 1.3763733782793243e-10, 1.122965000394129e-10, 8.630004494040329e-11, 6.246987936497669e-11, 4.259367933434496e-11, 2.735485965750919e-11, 1.6547739264812904e-11, 9.428827532184425e-12, 5.060478205607784e-12, 2.558234019908819e-12, 1.2181585715370543e-12, 5.463641878827953e-13, 2.308210803059994e-13, 9.185092994